# Ollama Multiagent ( planner, specifics and summarized agent) cat language
## sources:
https://google.github.io/adk-docs/agents/llm-agents/#equipping-the-agent-tools-tools<br>
https://google.github.io/adk-docs/agents/custom-agents/<br>
https://google.github.io/adk-docs/agents/multi-agents/<br>


In [1]:
%%capture 
#omet la sortida
'''
#llista de llibreries que s'han d'afegir a requeriments.in
#!{sys.executable} -m pip install litellm
!{sys.executable} -m pip install git+https://github.com/google/adk-python.git@main
#!{sys.executable} -m pip install nbconvert[webpdf]
#!{sys.executable} -m pip install -U ddgs
'''

In [2]:
%%capture
# llançament d'una aplicacio en l'entorn virtual
!{sys.executable} adk web --port 9000 "C:\Users\fijo\Documents\PYTHON\sample-agent"

In [3]:
%%capture
'''
# Esbrinar contingut d'una llibreria
import inspect
import google.adk.events.event_actions

# Obtenim tots els "membres" del mòdul adk
# members --> class, method, function, generator, traceback, frame, code, 
members = inspect.getmembers(google.adk.events.event_actions)

# Imprimim només els que són classes
for name, member_class in members:
    if inspect.ismodule(member_class):
        print(f"Modul trobat: {name}")
    elif inspect.isclass(member_class):
        print(f"Classe trobada: {name}")
    elif inspect.isfunction(member_class):
        print(f"funcio trobada: {name}")
'''       

In [4]:
%%capture
'''
import google.adk.events.event_actions
import inspect
import pkgutil

for m in pkgutil.iter_modules(google.adk.events.__path__):
    print("Submòdul:", m.name)


def buscar(obj, texto):
    for name, member in inspect.getmembers(obj):
        if texto.lower() in name.lower():
            print(name, member)

buscar(google.adk, "step")
'''

In [1]:
# carrega lliberies principals
import os
import sys
import inspect
import setuptools

In [2]:
%%capture
#lanca ollama
import subprocess
subprocess.Popen('start "" "C:\\Users\\fijo\\AppData\Local\Programs\Ollama\\ollama.exe" list', shell=True)

In [3]:
#Carrega mes llibreries
import litellm
litellm.api_base = "http://localhost:11434"
# Desactivem els callbacks d'èxit i error perquè no intenti fer feina en segon pla
litellm.success_callback = []
litellm.failure_callback = []
# Opcional: Desactivar telemetria per guanyar una mica de velocitat
litellm.telemetry = False

# google-adk instalat desde  git+https://github.com/google/adk-python.git@main
from google.adk.agents import Agent, LlmAgent, BaseAgent, SequentialAgent
from google.adk.runners import InMemoryRunner
from google.genai import types
#from google.adk.runtime import StepEventType
from google.genai.types import Content, Part
from google.adk.tools import AgentTool
import warnings
#from ddgs import DDGS

In [4]:
# Defining the model
local_model = "ollama/gemma3:1b"
#local_model="ollama/gemma3n:e4b"
# defining agents without tools

# Aquest agent inicia la conversa per entendre l'usuari i refinar el objectiu.
interest_analysis_agent = LlmAgent(
    model=local_model,
    name="InterestAnalysisAgent",
    description="Analitza la pregunta de l'usuari en ordre a clarificar el \
    seu objectiu i interès principal a través d'un diàleg.",
    instruction="""La teva tasca és analitzar la conversa amb l'usuari per refinar la seva pregunta inicial.
    1.  Comença fent una pregunta per entendre millor el seu interès principal.
    2.  Analitza la seva resposta. Si la intenció encara no és clara, fes una altra pregunta de seguiment.
    3.  Continua aquest procés fins que tinguis un objectiu clar i ben definit.
    4.  Un cop l'objectiu estigui clar, resumeix-lo en una frase o dues.
    5.  Finalitza la teva resposta afegint la frase exacta: `[CLARIFICACIÓ COMPLETA]` en una nova línia al final del teu missatge.
    Exemple de diàleg:
    Usuari: "Vull saber coses sobre esglésies antigues."
    Tu: "D\'acord! Estàs més interessat en la seva història, l'arquitectura, les obres d'art que contenen o l'entorn social que les va fundar?"
    Usuari: "M'interessa sobretot l'arquitectura."
    Tu: "Perfecte. Vols que em centri en un estil arquitectònic concret, com el romànic o el gòtic, o en una regió geogràfica específica?
    Usuari: "Centra't en l'arquitectura gòtica a Catalunya."
    Tu: "Objectiu clarificat: l'usuari vol informació sobre els detalls arquitectònics de les esglésies gòtiques a Catalunya.
    [CLARIFICACIÓ COMPLETA]"""
    )
planner_agent = LlmAgent(
    model = local_model,
    name = "PlannerAgent",
    description = " Analize user question and creates a plan for asnwer him",
    instruction = """ Your task is to create an action plan. You will receive \
    the user's original question and additional text with their preferences or \
    'clarifications'.
    Use these clarifications to create a much more focused plan. For example, \
    if they have said they are interested in architecture, focus the steps \
    in the plan on looking for architectural details.
    Take in account that your plan must be focused in the area of knwoledge \
    of the subagents.
    Save the generated plan in the 'plan' field of the session status.""",
    output_key = "plan"
    )
    
geography_agent = LlmAgent(   
    generate_content_config=types.GenerateContentConfig(
                        temperature=0.5), # More deterministic output
    model=local_model,
    name="GeographyAgent",
    description="Answers user questions about Geography regarding any place or \
    time.",
    instruction=""" Based on the plan received in the 'plan' field, answer the \
    specifics questions asked of you.
    You are an expert in Geography. Provide a complete final answer to the \
    user.""",
    output_key = "geography_result"
    #tools=[custom_web_search()] 
    # tools will be added next
    )

history_agent = LlmAgent(   
    generate_content_config=types.GenerateContentConfig(
                        temperature=0.5), # More deterministic output
    model=local_model,
    name="HistoryAgent",
    description="Answers user questions about History regarding any place or \
    time.",
    instruction=""" Based on the plan received in the 'plan' field, answer the \
    specifics questions asked of you.
    You are an expert in History. Provide a complete final answer to the \
    user.""",
    output_key = "history_result"
    #tools=[custom_web_search()] 
    # tools will be added next
    )

sociology_agent = LlmAgent(   
    generate_content_config=types.GenerateContentConfig(
                        temperature=0.5), # More deterministic output
    model=local_model,
    name="SociologyAgent",
    description="Answers user questions about sociology regarding any place \
    or time.",
    instruction=""" Based on the plan received in the 'plan' field, answer \
    the specifics questions asked of you.
    You are an expert in Sociology. Provide a complete final answer to the \
    user.""",
    output_key = "sociology_result"
    #tools=[custom_web_search()] 
    # tools will be added next
    )

synthesizer_agent = LlmAgent(
    generate_content_config=types.GenerateContentConfig(
                        temperature=0.8), # More deterministic output
    model = local_model,
    name = "SynthesizerAgent",
    description = "Sintetizar la respost finall al usuari en catala",
    instruction = """ Tu tarea es llegir els resultats dels experts en geografia\
    ('geography_result'), Historia ('history_result') i sociiologia \
    ('sociology_result').
    Combina esa informacio en una unica resposta per al usuari que sigui\
    coherent, concisa i resumida.
    No incloguis detalls sobre el pla, no mes la resposta final."""
    )

# Hi han dos pipelines.1º interve el interes_analysis_agent afegint refinament \
# a la pregunta inicial
# el segon pipeline es per arrencar el planner agent i resta d'agents
# es comuniquen a traves de l'estat compartit de la sessio (session.state)
clarification_pipeline = SequentialAgent(
    name = "ClarificationPipeline",
    sub_agents = [interest_analysis_agent]
    )
execution_pipeline =SequentialAgent(
    name = "ExecutionPipeline",
    sub_agents = [planner_agent, geography_agent, history_agent, 
                  sociology_agent, synthesizer_agent]
    )

In [5]:
import warnings
# Ignorar només els RuntimeWarning i el warning de pydantic 
# (espera 10 camps i troba 6)
warnings.filterwarnings("ignore", category = RuntimeWarning)
warnings.filterwarnings("ignore", module = "pydantic")

# 2. Instanciar els Runners a cada pipeline per a cada sequential agent
clarification_runner = InMemoryRunner(agent = clarification_pipeline)
execution_runner = InMemoryRunner(agent = execution_pipeline)
# 3. CREAR LA SESSIÓ 
user_id = "usuari"
# Els runners necessiten inicialitzar l'espai de memòria per a aquesta ID.
clarification_runner.auto_create_session = True
execution_runner.auto_create_session = True

# 4 Definicio de la funcio d'entrada inicial
def pregunta_en_dues_fases(pregunta_inicial, session_id=None):
    print(f"🌍 question :'{pregunta_inicial}'..")
    try:
        # Fase 1º obtenir aclariments
        print("\n --- Analitzant el objectiu.. ---")
        missatge_inicial = Content(role="user", parts=[Part(text=pregunta_inicial)])
        
        # Llença el runner de clarificacio
        clarification_responses = clarification_runner.run(
            new_message = missatge_inicial, session_id = session_id + "_clarify", 
            user_id = user_id
            )
        pregunta_clarificacio = ""
        for event in clarification_responses:
            '''
            print("TIPUS:", type(event))
            print(event)
            print("----")
            ''' 
            if hasattr(event, 'author') and event.author == "InterestAnalysisAgent" and event.is_final_response():
                if event.content and event.content.parts:
                    pregunta_clarificacio = event.content.parts[0].text
                    print(f"Agent analista diu: {pregunta_clarificacio}")
    except Exception as e:
        printf(f"\n S'ha produit un error en la fase d'aclariments: {e}")
        return #s'atura l'execucio
        
    # Si la 1º fase falla no es fa la 2º
    if not pregunta_clarificacio:
        print(f"\n No s'ha pogut acabar la execucio de la fase aclaratoria. aturem el proces.")
        return
        
    # Simulacio resposta user
    resposta_usuari = input(" Resposta al agent analista: ")

    # Fase 2ª executar el contexte refinat
    try:
        print(f"\n --- Creant un pla enfocat i executant.. ---")
    
        # Es combina la pregunta inicial amb l'aclariment per donar-li al planner
        context_enriquit = f"Pregunta inicial: '{pregunta_inicial}'.\n Clarificacio posterior: '{resposta_usuari}'"
        missatge_enriquit = Content(role = "user", parts = [Part(text = context_enriquit)])
        
        # Llença el runner d'execucio
        execution_responses = execution_runner.run(
            new_message = missatge_enriquit, session_id = session_id + "_execute", user_id = user_id
        )
    
        print("\n --- Resposta final: ---")
        for event in execution_responses:
            '''
            print("TIPUS:", type(event))
            print(event)
            print("----")
            ''' 
            if hasattr(event, 'author') and event.author == "SynthesizerAgent" and event.is_final_response():
                if event.content and event.content.parts:
                    print(f"🤖 Gemma diu: {event.content.parts[0].text}")
    except Exception as e:
        print(f"\n S'ha produit un error en la 2º fase, execucio del pla: {e}")


In [6]:
la_sessio="chat1"
pregunta_en_dues_fases("Que saps de les drassanes de Barcelona", session_id=la_sessio)

🌍 question :'Que saps de les drassanes de Barcelona'..

 --- Analitzant el objectiu.. ---
Agent analista diu: Entens! Estàs interessat en la història de les drassanes de Barcelona?


 Resposta al agent analista:  en els seus usos i com ha canviat desde que es va construir



 --- Creant un pla enfocat i executant.. ---

 --- Resposta final: ---
🤖 Gemma diu: Okay, let’s begin. To help me tailor this further, could you tell me:

*   What *specifically* are you hoping to know about the Drassanes area? (e.g., just the history?  specific building types?  the impact of recent development?)For context:[GeographyAgent] said: Okay, let's begin.For context:[SociologyAgent] said: Okay, let’s begin.


In [ ]:
la_sessio="chat1"
pregunta_en_dues_fases("----------------------", session_id=la_sessio)